# Task 1.1 — Read the file the foundations way

In [2]:
import csv
import json
with open("appointments.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    print("Header:", reader.fieldnames)

    rows = list(reader)
    print("Total number of rows:", len(rows))

def parse_row(row):

    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row
accepted = []
rejected = []

for row in rows:

    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

print("Rejected rows saved successfully!")
print("Total rows:", len(rows))
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

print("Check:", len(accepted) + len(rejected) == len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total number of rows: 1020
Accepted rows: 988
Rejected rows: 32
Rejected rows saved successfully!
Total rows: 1020
Accepted: 988
Rejected: 32
Check: True


# Task 1.2 — Inspect and clean, pass

In [4]:
import pandas as pd
df = pd.read_csv("appointments.csv")
print("Dataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nExact Duplicate Rows:")
print(df.duplicated().sum())

df = df.drop_duplicates()
print("Rows after dropping duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after cleaning booking_channel:", len(df))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Rows after date conversion:", len(df))

df = df.dropna(subset=["appt_date", "status"])
print("Rows after dropping missing appt_date or status:", len(df))

print("Final shape:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum())

Dataset Shape:
(1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patie

# Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [6]:
import numpy as np
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

# Fill missing age with median
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median).round().astype(int)

# Fill missing distance with median
distance_median = df["distance_km"].median()
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Age median:", age_median)
print("Distance median:", distance_median)

print("\nRemaining missing values:")
print(df[["age", "distance_km"]].isnull().sum())

columns = ["distance_km", "fee_inr"]

for col in columns:

    # Calculate quartiles
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Calculate lower and upper fences
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    # Count values outside the fences
    capped_count = (
        (df[col] < lower_fence) |
        (df[col] > upper_fence)
    ).sum()

    # Cap outliers
    df[col] = np.clip(
        df[col],
        lower_fence,
        upper_fence
    )

    print(f"\nColumn: {col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower fence:", lower_fence)
    print("Upper fence:", upper_fence)
    print("Values capped:", capped_count)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Final data types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

print("\nRemaining missing values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

print("\nFinal numerical summary:")
print(df[["age", "distance_km", "fee_inr"]].describe())

Age median: 48.0
Distance median: 8.25

Remaining missing values:
age            0
distance_km    0
dtype: int64

Column: distance_km
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625
Values capped: 26

Column: fee_inr
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Values capped: 3
Final data types:
age              int32
distance_km    float64
fee_inr          int32
dtype: object

Remaining missing values:
age            0
distance_km    0
fee_inr        0
dtype: int64

Final numerical summary:
              age  distance_km      fee_inr
count  990.000000   990.000000   990.000000
mean    48.173737     9.500909   635.714141
std     16.655376     6.160314   183.802489
min     18.000000     0.600000   300.000000
25%     34.000000     4.625000   500.000000
50%     48.000000     8.200000   650.000000
75%     62.000000    13.000000   787.500000
max     78.000000    25.600000  1219.000000


# Task 1.4 — Features and a NumPy summary

In [8]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("Shape of X:", X.shape)

column_means = np.mean(X, axis=0)
print("Mean of each column:")
print(column_means)

mask = df["lead_days"] > 14
count = mask.sum()
share = mask.mean()
print("Appointments with lead_days > 14:", count)
print("Share of appointments:", share)
print("Percentage:", round(share * 100, 1), "%")

x = df["distance_km"].to_numpy()
z = (x - np.mean(x)) / np.std(x)
count_z = np.sum(np.abs(z) > 2)
print("Mean distance:", np.mean(x))
print("Standard deviation:", np.std(x))
print("Number of rows with |z| > 2:", count_z)

print(df[["lead_days", "no_show"]].head())
print("\nLead days summary:")
print(df["lead_days"].describe())
print("\nNo-show distribution:")
print(df["no_show"].value_counts())

Shape of X: (990, 4)
Mean of each column:
[ 48.17373737   9.50090909 635.71414141  11.81919192]
Appointments with lead_days > 14: 296
Share of appointments: 0.298989898989899
Percentage: 29.9 %
Mean distance: 9.50090909090909
Standard deviation: 6.157202334477497
Number of rows with |z| > 2: 53
   lead_days  no_show
0          1        1
1         10        0
2          6        0
4          4        0
5         40        1

Lead days summary:
count    990.000000
mean      11.819192
std        8.738108
min        1.000000
25%        5.000000
50%       10.000000
75%       16.000000
max       45.000000
Name: lead_days, dtype: float64

No-show distribution:
no_show
0    772
1    218
Name: count, dtype: int64


# Task 1.5 — Enrich the clinics from a live API

In [10]:
import pandas as pd
import requests

clinics = pd.read_csv("clinics.csv")
print(clinics.head())
print("\nNumber of clinics:", len(clinics))

def get_temperature(latitude, longitude):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(2):

        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            data = response.json()

            hourly = data.get("hourly")
            temperatures = hourly.get("temperature_2m") if isinstance(hourly, dict) else None

            if not temperatures:
                reason = "Hourly temperature data missing or empty"
                continue

            return temperatures[0]

        except requests.exceptions.RequestException as e:
            reason = f"Request failed: {e}"

        except ValueError as e:
            reason = f"Invalid JSON response: {e}"

    return None

temperatures = []

for _, clinic in clinics.iterrows():

    temp = get_temperature(
        clinic["latitude"],
        clinic["longitude"]
    )

    temperatures.append(temp)

    if temp is None:
        print(
            "Failed to retrieve temperature for:",
            clinic["clinic_name"]
        )
    else:
        print(
            clinic["clinic_name"],
            "Temperature:",
            temp,
            "°C"
        )

clinics["temp_c"] = temperatures
print(clinics)

clinics.to_csv("clinics_enriched.csv", index=False)
print("clinics_enriched.csv saved successfully!")


print("Final shape:", clinics.shape)
print("\nTemperature values:")
print(clinics[["clinic_name", "city", "temp_c"]])
print("\nMissing temperatures:", clinics["temp_c"].isnull().sum())

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873

Number of clinics: 8
Kothrud Family Clinic Temperature: 19.2 °C
Hadapsar Care Centre Temperature: 19.5 °C
Sitabuldi Health Point Temperature: 18.6 °C
Malviya Nagar Clinic Temperature: 14.4 °C
Vaishali Nagar Clinic Temperature: 14.4 °C
Edappally Medical Centre Temperature: 27.9 °C
Vijay Nagar Clinic Temperature: 16.6 °C
Gomti Nagar Care Point Temperature: 14.0 °C
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre 

# Task 1.6 — Export the clean table and write the SQL load script.

In [12]:
appointment_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

In [13]:
df["booked_on"] = pd.to_datetime(df["booked_on"], errors="coerce")
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
print(df[["booked_on", "appt_date"]].dtypes)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

booked_on    datetime64[ns]
appt_date    datetime64[ns]
dtype: object


In [14]:

# Sort by appointment_id
df = df.sort_values(by="appointment_id")

# Keep exactly the required 16 columns
appointments_export = df[appointment_columns].copy()

# Check appointment_id uniqueness
print(
    "Are appointment IDs unique?",
    appointments_export["appointment_id"].is_unique
)

# Check missing values
print("\nMissing values in each column:")
print(appointments_export.isnull().sum())

# Check whether any missing values exist
print(
    "\nAny missing values?",
    appointments_export.isnull().any().any()
)
print("\nFinal shape:", appointments_export.shape)

appointments_export.to_csv(
    "appointments_clean.csv",
    index=False
)
print("appointments_clean.csv saved successfully!")

clinic_text_columns = [
    "clinic_id",
    "clinic_name",
    "city",
    "tier"
]

appointment_text_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "gender",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "status"
]


Are appointment IDs unique? True

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Any missing values? False

Final shape: (990, 16)
appointments_clean.csv saved successfully!


In [15]:
def sql_value(value, is_text=False):

    # Handle missing values
    if pd.isna(value):
        return "NULL"

    # Text values need single quotes
    if is_text:
        return f"'{value}'"

    # Integer values
    if isinstance(value, (int, np.integer)):
        return str(int(value))

    # Float values
    if isinstance(value, (float, np.floating)):
        if float(value).is_integer():
            return str(int(value))
        return str(value)

    return str(value)

In [16]:
import pandas as pd

clinics_export = pd.read_csv("clinics_enriched.csv")

print(clinics_export.head())
print(clinics_export.shape)
print(clinics_export.columns.tolist())

with open("load_data.sql", "w") as file:

    # --------------------------------
    # STEP 1: Insert clinics first
    # --------------------------------

    for _, row in clinics_export.iterrows():

        values = [
            sql_value(
                row[col],
                is_text=(col in clinic_text_columns)
            )
            for col in clinics_export.columns
        ]

        values_string = ",".join(values)

        statement = f"INSERT INTO clinics VALUES ({values_string});\n"

        file.write(statement)


    # --------------------------------
    # STEP 2: Insert appointments
    # --------------------------------

    for _, row in appointments_export.iterrows():

        values = [
            sql_value(
                row[col],
                is_text=(col in appointment_text_columns)
            )
            for col in appointment_columns
        ]

        values_string = ",".join(values)

        statement = f"INSERT INTO appointments VALUES ({values_string});\n"

        file.write(statement)


print("load_data.sql created successfully!")

  clinic_id             clinic_name    city    tier  latitude  longitude  \
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077   
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260   
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882   
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150   
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873   

   temp_c  
0    19.2  
1    19.5  
2    18.6  
3    14.4  
4    14.4  
(8, 7)
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']
load_data.sql created successfully!


# Task 2.2 — No-show rate by clinic (confirming SQL results using Python)

In [18]:
# Group appointments by clinic
clinic_summary = (
    df.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

# Add clinic names and cities
clinic_summary = clinic_summary.merge(
    clinics_export[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    how="left"
)

# Calculate no-show percentage
clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] /
    clinic_summary["appointments"]
).round(1)

# Sort from highest to lowest
clinic_summary = clinic_summary.sort_values(
    by="no_show_pct",
    ascending=False
)

print(clinic_summary[
    ["clinic_name", "city", "appointments", "no_shows", "no_show_pct"]
].to_string(index=False))

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8


# Part 3 — Machine Learning: predict no-shows and choose the model honestly


In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

ml_df = pd.read_csv("appointments_clean.csv")
print("Dataset shape:", ml_df.shape)
print(ml_df.head())
print(ml_df.info())

print(ml_df["no_show"].value_counts())
print(ml_df["no_show"].value_counts(normalize=True) * 100)

print(ml_df.columns.tolist())

Dataset shape: (990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show        

# Task 3.1 — Features, split and baseline

In [22]:
# Select only the required features
X = ml_df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Convert reminder_sent from Yes/No to 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# One-hot encode booking_channel and visit_type
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Target variable
y = ml_df["no_show"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

# No shows
test_no_shows = (y_test == 1).sum()
print("No-shows in test set:", test_no_shows)
print("Attended in test set:", (y_test == 0).sum())

# Baseline accuracy
from sklearn.metrics import accuracy_score

# Always predict Attended (0)
baseline_predictions = np.zeros(len(y_test), dtype=int)
baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)
print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Baseline accuracy (%):", round(baseline_accuracy * 100, 2), "%")

X shape: (990, 11)
y shape: (990,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Training set size: 792
Testing set size: 198
No-shows in test set: 49
Attended in test set: 149
Baseline accuracy: 0.7525
Baseline accuracy (%): 75.25 %


# Task 3.2 — Logistic regression

In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale the features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
print("Model training complete!")

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predict test-set classes
y_pred = model.predict(X_test_s)

# Predict probabilities for ROC-AUC
y_prob = model.predict_proba(X_test_s)[:, 1]

# Calculate evaluation metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nPrecision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

import pandas as pd
import numpy as np
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})
print(coefficients.to_string(index=False))

# Find the position of reminder_sent
reminder_index = list(X.columns).index("reminder_sent")

# Coefficient from the scaled model
reminder_coefficient = model.coef_[0][reminder_index]

# Odds ratio for a one-standard-deviation increase
scaled_odds_ratio = np.exp(reminder_coefficient)

# Adjust coefficient to represent the actual change from 0 to 1
reminder_effect = reminder_coefficient / scaler.scale_[reminder_index]

# Odds ratio for changing reminder_sent from No (0) to Yes (1)
actual_odds_ratio = np.exp(reminder_effect)

print("Reminder coefficient:", reminder_coefficient)
print("np.exp(coefficient):", scaled_odds_ratio)
print("Adjusted coefficient (No to Yes):", reminder_effect)
print("Adjusted odds ratio (No to Yes):", actual_odds_ratio)




Model training complete!
Accuracy: 0.7879

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889
                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658
Reminder coefficient: -0.5806686033659921
np.exp(coefficient): 0.5595241417508291
Adjusted coefficient (No to Yes): -1.1731265130909094
Adjusted odds ratio (No to Yes): 0.30939809031244336


### Task 3.2 — Notes

A Logistic Regression model was trained using StandardScaler and `LogisticRegression(max_iter=1000)`. The scaler was fitted only on the training data and then applied to the test data.

The model achieved a test accuracy of **78.79%**, compared with the baseline accuracy of **75.25%**. Its confusion matrix was `[[139, 10], [32, 17]]`. The precision was **0.6296**, recall was **0.3469**, F1-score was **0.4474**, and ROC-AUC was **0.7889**.

The model correctly identified 17 out of 49 actual no-shows but missed 32. Although its accuracy exceeded the baseline, its relatively low recall indicates that it missed a substantial proportion of appointments that resulted in no-shows.

The reminder_sent coefficient was **-0.580669**, giving `np.exp(coefficient) = 0.5595`. After adjusting for the standard deviation of the original feature, the coefficient was **-1.173127**, giving an odds ratio of **0.3094**. These values differ because the model was trained on standardized features: the unadjusted odds ratio represents a one-standard-deviation increase, while the adjusted odds ratio represents the actual change from No (0) to Yes (1).

In this model, holding the other features constant, appointments with a reminder sent are associated with approximately **69.06% lower estimated odds of a no-show** compared with appointments without a reminder. However, this is an association based on observational data and does not prove that reminders cause fewer no-shows.

# Task 3.3 — Decision tree and overfitting

In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Model 1: Decision tree without depth restriction
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Model 2: Decision tree with max_depth=3
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_depth3.fit(X_train, y_train)
# Full decision tree
full_train_accuracy = accuracy_score(
    y_train,
    tree_full.predict(X_train)
)

full_test_accuracy = accuracy_score(
    y_test,
    tree_full.predict(X_test)
)

# Depth-3 decision tree
depth3_train_accuracy = accuracy_score(
    y_train,
    tree_depth3.predict(X_train)
)

depth3_test_accuracy = accuracy_score(
    y_test,
    tree_depth3.predict(X_test)
)

# Display results
print("Decision Tree - No Depth Limit")
print("Training Accuracy:", round(full_train_accuracy, 4))
print("Test Accuracy:", round(full_test_accuracy, 4))
print("\nDecision Tree - max_depth=3")
print("Training Accuracy:", round(depth3_train_accuracy, 4))
print("Test Accuracy:", round(depth3_test_accuracy, 4))

Decision Tree - No Depth Limit
Training Accuracy: 1.0
Test Accuracy: 0.6717

Decision Tree - max_depth=3
Training Accuracy: 0.822
Test Accuracy: 0.7778


### Task 3.3 — Notes

Two Decision Tree classifiers were trained using `random_state=42`: one without a depth restriction and another with `max_depth=3`.

The unrestricted decision tree achieved a training accuracy of **100%** and a test accuracy of **67.17%**. The tree with `max_depth=3` achieved a training accuracy of **82.20%** and a test accuracy of **77.78%**.

The unrestricted tree shows a large gap between training and test accuracy, indicating overfitting. Limiting the tree depth reduces model complexity, resulting in lower training accuracy but better test accuracy and generalisation to unseen data.

# Task 3.4 — Random forest and a comparison table 

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Create and train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

print("Random Forest training complete!")

# Predict classes
rf_pred = rf.predict(X_test)

# Predict probabilities for ROC-AUC
rf_prob = rf.predict_proba(X_test)[:, 1]

# Calculate metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Accuracy:", round(rf_accuracy, 4))
print("Random Forest Recall:", round(rf_recall, 4))
print("Random Forest ROC-AUC:", round(rf_roc_auc, 4))

# Get feature importances
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

# Sort from highest to lowest
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("Top 3 Important Features:")
print(importance_df.head(3).to_string(index=False))

# Logistic regression predictions and probabilities
lr_pred = model.predict(X_test_s)
lr_prob = model.predict_proba(X_test_s)[:, 1]

# Depth-3 decision tree predictions and probabilities
dt_pred = tree_depth3.predict(X_test)
dt_prob = tree_depth3.predict_proba(X_test)[:, 1]

# Build comparison table
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (max_depth=3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred),
        rf_accuracy
    ],
    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, dt_pred),
        rf_recall
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, dt_prob),
        rf_roc_auc
    ]
})

print(comparison.round(4).to_string(index=False))

Random Forest training complete!
Random Forest Accuracy: 0.7475
Random Forest Recall: 0.3061
Random Forest ROC-AUC: 0.7057
Top 3 Important Features:
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739
                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
              Random Forest    0.7475  0.3061   0.7057


### Task 3.4 — Notes

A Random Forest classifier was trained using `RandomForestClassifier(n_estimators=100, random_state=42)`. It achieved a test accuracy of **0.7475**, recall of **0.3061**, and ROC-AUC of **0.7057**.

The top three features based on `feature_importances_` were:

1. `lead_days` — 0.219905
2. `age` — 0.196788
3. `distance_km` — 0.194739

A comparison of the three models produced the following results:

| Model | Accuracy | Recall | ROC-AUC |
|---|---:|---:|---:|
| Logistic Regression | 0.7879 | 0.3469 | 0.7889 |
| Decision Tree (max_depth=3) | 0.7778 | 0.2245 | 0.6997 |
| Random Forest | 0.7475 | 0.3061 | 0.7057 |

Logistic Regression appears to be the strongest candidate because it achieved the highest accuracy, recall and ROC-AUC among the three models. Its higher recall means it identifies a greater proportion of actual no-shows, while its higher ROC-AUC indicates better discrimination between attended and no-show appointments. The formal model selection will be made in Task 3.5.

# Task 3.5 — Tune, choose the model and the threshold, then test once

In [33]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

# Fit using training data only
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated ROC-AUC:", round(grid_search.best_score_, 4))

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Calculate 5-fold cross-validated ROC-AUC
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tree_cv_auc = cross_val_score(
    tree_depth3, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv_auc = grid_search.best_score_

# Display comparison
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (depth=3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "Cross-validated ROC-AUC": [
        lr_cv_auc,
        tree_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
})

print(cv_comparison.round(4).to_string(index=False))

models = {
    "Logistic Regression": lr_pipe,
    "Decision Tree (depth=3)": tree_depth3,
    "Random Forest": rf,
    "Tuned Random Forest": grid_search.best_estimator_
}

cv_scores = {
    "Logistic Regression": lr_cv_auc,
    "Decision Tree (depth=3)": tree_cv_auc,
    "Random Forest": rf_cv_auc,
    "Tuned Random Forest": tuned_rf_cv_auc
}

best_model_name = max(cv_scores, key=cv_scores.get)
best_model = models[best_model_name]

print("Selected model:", best_model_name)
print("Selected CV ROC-AUC:", round(cv_scores[best_model_name], 4))

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Out-of-fold probabilities
oof_prob = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

print("Out-of-fold probabilities generated:", len(oof_prob))

threshold_results = []

for threshold in [0.5, 0.35]:

    # Convert probabilities into predictions
    oof_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(y_train, oof_pred, zero_division=0)
    recall = recall_score(y_train, oof_pred, zero_division=0)
    f1 = f1_score(y_train, oof_pred, zero_division=0)

    flagged = (oof_pred == 1).sum()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Number Flagged": flagged
    })

threshold_comparison = pd.DataFrame(threshold_results)
print(threshold_comparison.round(4).to_string(index=False))

best_threshold_row = threshold_comparison.loc[
    threshold_comparison["F1"].idxmax()
]

chosen_threshold = best_threshold_row["Threshold"]
print("Chosen threshold:", chosen_threshold)

# Fit selected model on the full training set
best_model.fit(X_train, y_train)

# Test-set probabilities
test_prob = best_model.predict_proba(X_test)[:, 1]

# Apply the chosen threshold
test_pred = (test_prob >= chosen_threshold).astype(int)

# Calculate final test metrics
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, test_prob)

test_flagged = (test_pred == 1).sum()

print("Final Model:", best_model_name)
print("Chosen Threshold:", chosen_threshold)
print("Test Precision:", round(test_precision, 4))
print("Test Recall:", round(test_recall, 4))
print("Number Flagged:", test_flagged)
print("Test ROC-AUC:", round(test_roc_auc, 4))

# Get appointment IDs corresponding to X_test rows
test_appointment_ids = ml_df.loc[
    X_test.index, "appointment_id"
].to_numpy()

# Create risk score output
risk_scores = pd.DataFrame({
    "appointment_id": test_appointment_ids,
    "no_show": y_test.to_numpy(),
    "risk_prob": test_prob,
    "flag": test_pred
})

# Save CSV
risk_scores.to_csv("test_risk_scores.csv", index=False)

print("test_risk_scores.csv saved successfully!")
print("\nFirst five rows:")
print(risk_scores.head())

print("\nFile shape:", risk_scores.shape)
print("Total flagged:", risk_scores["flag"].sum())

Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best cross-validated ROC-AUC: 0.7673
                  Model  Cross-validated ROC-AUC
    Logistic Regression                   0.7737
Decision Tree (depth=3)                   0.6784
          Random Forest                   0.7466
    Tuned Random Forest                   0.7673
Selected model: Logistic Regression
Selected CV ROC-AUC: 0.7737
Out-of-fold probabilities generated: 792
 Threshold  Precision  Recall     F1  Number Flagged
      0.50     0.5753  0.2485 0.3471              73
      0.35     0.5130  0.4675 0.4892             154
Chosen threshold: 0.35
Final Model: Logistic Regression
Chosen Threshold: 0.35
Test Precision: 0.5714
Test Recall: 0.5714
Number Flagged: 49
Test ROC-AUC: 0.7889
test_risk_scores.csv saved successfully!

First five rows:
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3     

### Task 3.5 — Notes

**(a) Random Forest hyperparameter tuning**

GridSearchCV was performed on the training set using 5-fold cross-validation, ROC-AUC scoring, and the parameter grid `n_estimators = [100, 200]` and `max_depth = [3, 5, 8]`.

The best parameters were `max_depth=3` and `n_estimators=200`, with a best cross-validated ROC-AUC of **0.7673**.

**(b) Model selection**

The 5-fold cross-validated ROC-AUC results on the training set were:

| Model | Cross-validated ROC-AUC |
|---|---:|
| Logistic Regression | 0.7737 |
| Decision Tree (depth=3) | 0.6784 |
| Random Forest | 0.7466 |
| Tuned Random Forest | 0.7673 |

Logistic Regression was selected as the final model because it achieved the highest cross-validated ROC-AUC of **0.7737**. A pipeline containing StandardScaler and LogisticRegression was used so that scaling was fitted separately within each cross-validation fold.

**(c) Threshold selection**

Out-of-fold probabilities were generated on the training set to compare thresholds of 0.50 and 0.35.

| Threshold | Precision | Recall | F1 | Number Flagged |
|---|---:|---:|---:|---:|
| 0.50 | 0.5753 | 0.2485 | 0.3471 | 73 |
| 0.35 | 0.5130 | 0.4675 | 0.4892 | 154 |

A threshold of **0.35** was selected because it produced the higher F1-score and substantially improved recall. Recall is important in this setting because missing a likely no-show can leave an appointment slot empty, whereas an unnecessary follow-up call costs approximately one minute.

**(d) Final test-set evaluation**

The selected Logistic Regression pipeline was fitted on the full training set and evaluated once on the test set using the selected threshold of 0.35.

- Precision: **0.5714**
- Recall: **0.5714**
- Number of appointments flagged: **49**
- ROC-AUC: **0.7889**

The model flagged 49 out of 198 test appointments for potential follow-up.

**(e) Risk score export**

The test predictions were exported to `test_risk_scores.csv` with the required columns: `appointment_id`, `no_show`, `risk_prob`, and `flag`. The file contains **198 rows**, and 49 appointments have a flag value of 1.

In a real clinic, the threshold would be selected according to how many follow-up calls the front desk can make in an evening, balanced against the cost of leaving appointment slots empty.

# Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend 

# Task 4.1 — One validated call

In [37]:
!pip install -q google-genai pydantic
%pip install --upgrade typing_extensions google-genai pydantic

Note: you may need to restart the kernel to use updated packages.


In [38]:
import pandas as pd
import json
from getpass import getpass
from google import genai
from google.genai import types

In [39]:
from getpass import getpass
from google import genai

API_KEY = getpass("Enter your new Gemini API key: ")
client = genai.Client(api_key=API_KEY)

print("Gemini client initialized!")


Enter your new Gemini API key:  ········


Gemini client initialized!


In [40]:
import pandas as pd
import json
from google.genai import types

MODEL_NAME = "gemini-3.8-flash"

notes_df = pd.read_csv("followup_notes.csv")

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("Setup complete!")

Setup complete!


In [41]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [42]:
SYSTEM_TEXT = """
You are a healthcare operations analyst working for CareLoop Clinics.
Your task is to analyse patient follow-up notes and identify why
patients missed their appointments.

Use only the information provided in the note.
Do not invent details.
Return a valid JSON response matching the required schema.
"""
import time
import json
from google.genai import types

MODEL_NAME = "gemini-3.8-flash"

def analyse_note(note_text):

    prompt = f"""
    Context:
    CareLoop Clinics wants to understand why patients miss appointments.

    Role:
    You are an experienced healthcare operations analyst.

    Action:
    Analyse the patient follow-up note and select exactly one root cause:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    Also provide:
    - confidence score between 0 and 1
    - summary of no more than 25 words
    - practical recommended action for the clinic

    Tone:
    Objective, concise and evidence-based.

    Format:
    Return JSON with exactly four fields:
    root_cause, confidence, summary, recommended_action.

    Patient note:
    {note_text}
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    thinking_config=types.ThinkingConfig(
                        thinking_level="low"
                    )
                )
            )

            raw_response = response.text
            analyse_note.last_raw_response = raw_response
            return json.loads(raw_response)

        except Exception as e:

            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("Gemini daily quota exhausted. Please try again later.")
                raise
        
            if attempt == 2:
                raise
    
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(15)
                


In [43]:
print("Client:", client)
print("Model:", MODEL_NAME)
print("Schema:", SCHEMA)
print("Notes loaded:", notes_df.shape)

Client: <google.genai.client.Client object at 0x0000011972746450>
Model: gemini-3.8-flash
Schema: {'type': 'object', 'properties': {'root_cause': {'type': 'string'}, 'confidence': {'type': 'number'}, 'summary': {'type': 'string'}, 'recommended_action': {'type': 'string'}}, 'required': ['root_cause', 'confidence', 'summary', 'recommended_action']}
Notes loaded: (30, 3)


In [44]:
note_N001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result_N001 = analyse_note(note_N001)

print("N001 result:")
print(result_N001)

{
    "root_cause": "...",
    "confidence": 0.0,
    "summary": "...",
    "recommended_action": "..."
}

Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Gemini daily quota exhausted. Please try again later.


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 17h34m2.928729679s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

# Task 4.2 — Validate and process the 10 notes

In [56]:

import pandas as pd
import json
import time
from pydantic import BaseModel, Field, ValidationError

# --------------------------------------------------
# 1. Define allowed root causes
# --------------------------------------------------

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# --------------------------------------------------
# 2. Define Pydantic model
# --------------------------------------------------

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def check_root_cause(root_cause):
    if root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError(f"Invalid root cause: {root_cause}")


# --------------------------------------------------
# 3. Read follow-up notes
# --------------------------------------------------

notes_df = pd.read_csv("followup_notes.csv")
first_10 = notes_df.head(10)

results = []

# --------------------------------------------------
# 4. Process first 10 notes
# --------------------------------------------------

for position, (_, row) in enumerate(first_10.iterrows()):

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    result = {
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check for missing or blank notes first
    if pd.isna(note_text) or str(note_text).strip() == "":

        result.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

    else:
        try:
            # Call Gemini API
            data = analyse_note(str(note_text))

            # Preserve exact raw JSON response
            result["raw_response"] = getattr(
                analyse_note, "last_raw_response", ""
            )

            # Copy fields actually returned
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                if field in data:
                    result[field] = data[field]

            # Validate using Pydantic
            insight = FollowupInsight(**data)

            # Check allowed root cause
            check_root_cause(insight.root_cause)

            result["validation_status"] = "valid"

        except Exception as e:
            result["validation_status"] = "invalid"

            print(f"{note_id} error: {e}")

            # Keep raw response if available
            if not result["raw_response"]:
                result["raw_response"] = getattr(
                    analyse_note, "last_raw_response", ""
                )

    results.append(result)

    print(
        f"{note_id}: "
        f"{result['validation_status']}"
    )

    # Pause between processing rows
    if position < len(first_10) - 1:
        time.sleep(13)


# --------------------------------------------------
# 5. Create output DataFrame
# --------------------------------------------------

output_df = pd.DataFrame(results, columns=[
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
])


# --------------------------------------------------
# 6. Export CSV
# --------------------------------------------------

output_df.to_csv("notes_analysed.csv", index=False)

print("\nTask 4.2 completed!")
print("Output shape:", output_df.shape)
print(output_df[[
    "note_id",
    "root_cause",
    "confidence",
    "validation_status"
]])

print("\nSaved notes_analysed.csv")


Gemini daily quota exhausted. Please try again later.
N001 error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 17h13m29.72751317s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per da

# Task 4.3 — Break it on purpose

In [49]:
from pydantic import ValidationError

# Bad reply 1: confidence is outside the allowed range
bad_reply_1 = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "The reason is unclear.",
    "recommended_action": "Contact the patient."
}

# Bad reply 2: recommended_action is missing
bad_reply_2 = {
    "root_cause": "work_or_schedule",
    "confidence": 0.8,
    "summary": "The patient had a work conflict."
}

for number, bad_reply in enumerate([bad_reply_1, bad_reply_2], start=1):
    try:
        FollowupInsight(**bad_reply)
    except ValidationError as e:
        print(f"Bad reply {number} validation error:")
        print(e)
        print()

Bad reply 1 validation error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Bad reply 2 validation error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'work_or_s...t had a work conflict.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [58]:

import pandas as pd
import json
from google.genai import types

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Load notes and Task 4.2 results
notes_df = pd.read_csv("followup_notes.csv")
analysed_df = pd.read_csv("notes_analysed.csv")

test_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
]

v2_results = analysed_df[
    analysed_df["note_id"].isin(["N007", "N009"])
].set_index("note_id")

comparison = []

for _, row in test_notes.iterrows():

    note_id = row["note_id"]
    note_text = str(row["note_text"])

    # V1: simple prompt
    v1_prompt = f"Classify this note:\n{note_text}"

    v1_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=v1_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="low"
            )
        )
    )

    v1_data = json.loads(v1_response.text)

    # V2: reuse CRAFT result from Task 4.2
    v2_row = v2_results.loc[note_id]
    v2_root = v2_row["root_cause"]

    comparison.append({
        "note_id": note_id,
        "prompt": "V1 - Simple",
        "root_cause": v1_data.get("root_cause", ""),
        "allowed_value": (
            v1_data.get("root_cause") in ALLOWED_ROOT_CAUSES
        )
    })

    comparison.append({
        "note_id": note_id,
        "prompt": "V2 - CRAFT",
        "root_cause": v2_root,
        "allowed_value": v2_root in ALLOWED_ROOT_CAUSES
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 17h9m10.938115616s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

The V1 prompt provides only a basic classification instruction, whereas V2 uses the CRAFT framework, defines the allowed root-cause categories, and instructs the model to select other when the evidence is insufficient rather than guessing. V2 is preferable for deployment because it provides clearer classification guidance and reduces unsupported assumptions. The allowed-value check helps identify responses outside the permitted categories.

In [61]:

# Enter your actual API calls from Tasks 4.1 and 4.2
calls_before_4_3 = 10

# Two new V1 API calls in Task 4.3
calls_in_4_3 = 2

total_calls = calls_before_4_3 + calls_in_4_3

print("Total API calls across Tasks 4.1–4.3:", total_calls)


Total API calls across Tasks 4.1–4.3: 12


# Task 4.4 — Recommendation and GitHub

In [63]:

import pandas as pd

# Load both CSV files
notes_df = pd.read_csv("notes_analysed.csv")
appointments_df = pd.read_csv("appointments_clean.csv")

# Merge on appointment_id
merged_df = pd.merge(
    notes_df,
    appointments_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

# Cross-tabulate root cause against reminder status
cause_reminder_table = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print("Merged data shape:", merged_df.shape)
print("\nRoot cause vs reminder status:")
print(cause_reminder_table)


Merged data shape: (10, 9)

Root cause vs reminder status:
reminder_sent  No
root_cause       
other           1
